<img align="left" src="https://panoptes-uploads.zooniverse.org/project_avatar/86c23ca7-bbaa-4e84-8d8a-876819551431.png" type="image/png" height=100 width=100>
</img>
<h1 align="right">Train ML models</h1>
<h3 align="right"><a href="https://colab.research.google.com/github/ocean-data-factory-sweden/kso/blob/main/notebooks/analyse/Train_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a></h3>
<h3 align="right">Written by the KSO Team</h3>

This notebook takes you through the process of importing a baseline model, training it on a dataset and evaluating the quality of the model. If you do not have a project with us yet, you can run the template project to get a taste of how it all works. This notebook assumes that the user has prepared the dataset for model training, see Tutorial #8 for details on the required setup.

🔴 <span style="color:red">&nbsp;NOTE: In order to run this notebook, you need to have a Weights and Biases account. If you want to become a member of our Koster team on Weights and Biases, you may request this access by contacting jurie.germishuys@combine.se. But this is not necessary to run the template project. </span>

In [ ]:
# Run this cell to set the directory correct
import sys
import os

kso_path = os.path.abspath(os.path.join(os.getcwd(), "../.."))
sys.path.insert(0, kso_path)
%load_ext autoreload
%autoreload 2

# Set config (NOTE: SHOULD BECOME PART OF A CONFIG FILE, HOW DO WE STRUCTURE THIS?)

In [ ]:
# Change these variables
project_name = "Template project"  # available projects can be found in "../kso_utils/db_starter/projects_list.csv"
data_path = "ml-template-data"
exp_name = "running_on_lumi"
model_name = "yolov8m-base-model"
model_download_dir = "."
batch_size = 1
epochs = 1
img_h = 128
conf_thres = 0.5

Tips for batch size and epochs: There are no strict rules for this and the best settings will depend on the choice of GPU and some randomness that we have encountered while training models. Therefore it will be some trial and error. As a starting point we advice to use a batch size of 8. For smaller datasets, we have experienced that 50-100 epochs has been sufficient to get good performance on the model (metrics that have reached a plateau), but to not overfit to the training set.

To choose a model, you can use the list below to find all the available baseline models at the registry you are using.

In [ ]:
from kso_utils.registries.wandb_registry import show_available_models

show_available_models("", baseline=True)
# TODO:remove the hardcoded registry, how to do this without having the mlp yet?

# Train the model

In [ ]:
from kso_utils.train_models import train_models

mlp = train_models(
    project_name,
    data_path,
    exp_name,
    model_name,
    model_download_dir,
    batch_size,
    epochs,
    img_h,
)

The model is now done with training. To see the loss, precision, recall and some other parameters per training epoch, click on the link in the previous cell. Here you can see your run in Weights and Biases.

For more evaluation, continue to the next cell.

# Evaluate the model

In [ ]:
mlp.eval_yolo(exp_name=exp_name, conf_thres=conf_thres)

As output from the cell above, you also see the standard evaluation from ultralytics printed: some numbers logged on the screen, and 3 files that are stored in the folder 'your_experiment_name'_val.

The numbers logged on the screen represent the following:
* The first 7 numbers are the: mean precision, mean recal, mean average precision calculated at IOU threshold 0.5 (map@0.5), the mean average precision calculated at different IOU thresholds of 0.5-0.95 with steps of 0.05 (map@0.5:0.95) and then 3 training losses based on predicting the box, object or class.
* The array gives the ap@0.5 per class.
* The last 3 numbers are the same as the numbers that are already printed in a line above, where it says 'Speed: … ms per....'

For a biological evaluation of the model, please see Evaluate_models.

# (Optional) : Enhance annotations using trained model

Enhancement uses the trained model to increase the amount of annotations in the training data. This should only be done in cases where it is absolutely necessary as bad predictions lead to worse predictions when used to train the next iteration of the model.


🔴 <span style="color:red">&nbsp;NOTE: We recommend using a relatively high confidence threshold when enhancing trained models as low confidence predictions could significantly impact the quality of your annotated data. This is currently only available for object detection models.  </span>

In [ ]:
eh_conf_thres = 0.5
project_path = ""  # ???

In [ ]:
mlp.enhance_yolo(
    in_path=mlp.output_path,  # IS THIS CORRECT??
    project_path=project_path,
    conf_thres=eh_conf_thres,
    img_size=[640, 640],
)

### Choose run to use as enhanced annotations

In [ ]:
runs = ""  # ???

In [ ]:
# Move enhanced annotations to original run folder (NB: This will replace the original annotations)
mlp.enhance_replace(runs)

#### Once you have moved the new labels to the original label location, you can train your model again.

In [ ]:
# END